# Appendix: Forward-Deployed Engineering

A forward-deployed engineer turns an ambiguous customer problem into a working, measurable system inside real operational constraints.


## What to learn

- Discovery identifies the user, workflow, pain point, data, and decision owner.
- A thin vertical slice tests the riskiest assumption with real users and data.
- Productionization adds security, reliability, integration, and ownership.
- Success metrics connect technical behavior to user or business outcomes.

## Decision rule

Do not start with a broad platform build. Define the decision and baseline, deliver one end-to-end workflow, measure adoption and quality, document ownership, and stop or redesign when the evidence does not support expansion.


## Runnable example

Run the following cells, then change one input and observe what changes. Focus on the decision being demonstrated rather than framework-specific syntax.


In [ ]:
# Import the dependencies used by this example.
from typing import Literal, TypedDict
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command

# Define the data shape and small operations before running them.
class Pilot(TypedDict):
    use_case: str
    baseline: float
    target: float
    measured: float
    decision: str

def scope(state: Pilot):
    if not state["use_case"] or state["target"] <= state["baseline"]:
        return {"decision": "invalid-scope"}
    return {"decision": "run-pilot"}

def gate(state: Pilot) -> Command[Literal["ship", "stop"]]:
    passed = state["measured"] >= state["target"]
    return Command(update={"decision": "ship" if passed else "stop"}, goto="ship" if passed else "stop")

# Configure the framework object; this line prepares it but may not execute it yet.
builder = StateGraph(Pilot)
builder.add_node("scope", scope)
builder.add_node("gate", gate)
builder.add_node("ship", lambda state: {})
builder.add_node("stop", lambda state: {})
builder.add_edge(START, "scope")
builder.add_edge("scope", "gate")
builder.add_edge("ship", END)
builder.add_edge("stop", END)
pilot_graph = builder.compile()
# Execute the configured model or workflow with the input below.
pilot_graph.invoke({"use_case": "support triage", "baseline": .62, "target": .80,
                    "measured": .84, "decision": "new"})


In [ ]:
"""Engagement scoping calculator -- the artifact an FDE produces after a
discovery workshop. No API needed -- pure-stdlib rubric scoring: rank
candidate agent use cases on value/feasibility/data-readiness/risk,
apply hard go/no-go gates, and print an eval-gated rollout timeline."""

# Import the dependencies used by this example.
from dataclasses import dataclass

WEIGHTS = {"value": 0.35, "feasibility": 0.25, "data": 0.25, "safety": 0.15}

@dataclass
# Define the data shape and small operations before running them.
class UseCase:
    name: str
    value: int        # 1-5: P&L impact if it works (5 = board-level line item)
    feasibility: int  # 1-5: can current models do this reliably in this workflow?
    data: int         # 1-5: clean, accessible, permissioned data exists
    risk: int         # 1-5: blast radius (5 = irreversible, customer-facing)
    sponsor: bool     # named executive owner who feels the baseline metric
    hitl: bool        # a human-in-the-loop checkpoint is workable
    baseline: str     # the metric the agent must beat ("" = unmeasured)

    def score(self) -> float:
        parts = {"value": self.value, "feasibility": self.feasibility,
                 "data": self.data, "safety": 6 - self.risk}
        return round(sum(WEIGHTS[k] * v for k, v in parts.items()), 2)

    def gates(self):
        """Hard go/no-go gates. Any failure blocks the pilot regardless of score."""
        return [
            ("Named executive sponsor", self.sponsor),
            ("Baseline metric instrumented (no baseline, no eval, no pilot)",
             bool(self.baseline)),
            ("Data readiness >= 3 (else run a data-plumbing sprint first)",
             self.data >= 3),
            ("Blast radius >= 4 requires a human-in-the-loop checkpoint",
             self.risk < 4 or self.hitl),
        ]

CANDIDATES = [
    UseCase("Tier-1 support deflection agent", 4, 4, 4, 4, True, True,
            "38% self-serve deflection; CSAT 4.1"),
    UseCase("Fully autonomous refund approvals", 5, 3, 3, 5, True, False,
            "$2.3M/yr manual refund-ops cost"),
    UseCase("Claims-intake document triage", 4, 5, 4, 2, True, True,
            "11 min median intake handling time"),
    UseCase("Exec 'ask-anything' BI chatbot", 2, 3, 2, 3, False, True, ""),
    UseCase("Sales-call summary -> CRM sync", 3, 5, 5, 1, True, True,
            "27% of calls logged with next steps"),
]

ROLLOUT = [
    ("Wk 0-2",   "Discovery + success-criteria workshop",
     "signed one-page scope; baseline metric instrumented"),
    ("Wk 2-4",   "Prototype on real customer data (inside their VPC)",
     "offline eval >= 90% on a frozen, versioned 100-case golden set"),
    ("Wk 4-8",   "Shadow mode: agent runs on live traffic, humans still act",
     "agreement with human decision >= 85%; zero sev-1 outputs"),
    ("Wk 8-10",  "Canary: 5% -> 25% of traffic, kill switch armed",
     "beats baseline metric; escalation < 10%; rollback drill passed"),
    ("Wk 10-13", "GA + handoff to the customer's team",
     "runbook, dashboards, on-call owned by customer; FDE exits"),
]

def main():
    ranked = sorted(CANDIDATES, key=lambda u: u.score(), reverse=True)
    print(f"{'Use case':<36} {'Val':>3} {'Fea':>3} {'Dat':>3} {'Rsk':>3} {'Score':>6}")
    print("-" * 62)
    for u in ranked:
        print(f"{u.name:<36} {u.value:>3} {u.feasibility:>3} {u.data:>3} "
              f"{u.risk:>3} {u.score():>6}")

    pilot = None
    print("\nGo/no-go gates (ranked order):")
    for u in ranked:
        failures = [label for label, ok in u.gates() if not ok]
        print(f"  [{'GO' if not failures else 'NO-GO':>5}] {u.name}")
        for f in failures:
            print(f"           blocked by: {f}")
        if not failures and pilot is None:
            pilot = u

    if pilot is None:
        print("\nNo candidate cleared the gates -> run a readiness sprint, not a pilot.")
        return
    print(f"\nRecommended pilot: {pilot.name}")
    print(f"  Baseline to beat: {pilot.baseline}")
    print(f"  Weighted score:   {pilot.score()} / 5.0  (note: not the highest-value")
    print(  "  idea -- the low-blast-radius, data-ready workflow wins the pilot slot)")
    print("\nEval-gated rollout plan (each gate blocks the next phase):")
    for week, phase, gate in ROLLOUT:
        print(f"  {week:<9} {phase}")
        print(f"  {'':<9}   GATE -> {gate}")
    print("\nKickback rule: if any gate slips past day 90, demote to standard")
    print("onboarding and re-enter discovery -- never drift into open-ended consulting.")

main()


## Study questions

1. What problem does this pattern solve?
2. What is the simplest alternative?
3. Which failure would tell you that the pattern is implemented incorrectly?
4. What measurement would justify adding more complexity?

## Before using it

- [ ] The requirement and success measure are explicit.
- [ ] Inputs and outputs are validated.
- [ ] Failures, retries, and stopping conditions are bounded.
- [ ] Security and privacy match the consequences of the action.
- [ ] The behavior is tested on realistic examples.
